In [22]:
import pandas as pd
import ast
import networkx as nx
import plotly.express as px

#### Loading Data

In [23]:
data_path = "../data/usc-x-24-us-election/part_1/"

files = [
    "may_july_chunk_1.csv.gz",
    "may_july_chunk_2.csv.gz",
    "may_july_chunk_3.csv.gz",
    "may_july_chunk_4.csv.gz",
    "may_july_chunk_5.csv.gz"
]

dfs = []
for f in files:
    temp = pd.read_csv(data_path + f, compression='gzip')
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Total tweets:", len(df))

Total tweets: 250000


#### Create Graph

In [24]:
print(df['mentionedUsers'].dropna().head(10))

0    [{'id_str': '1483596463221264387', 'name': 'Lu...
1    [{'id_str': '423966379', 'name': 'CONSERVATIVE...
2    [{'id_str': '1480159185606258692', 'name': 'Po...
3    [{'id_str': '254117355', 'name': 'Morning Joe'...
4                                                   []
5                                                   []
6    [{'id_str': '10228272', 'name': 'YouTube', 'sc...
7    [{'id_str': '428454304', 'name': 'Harry Sisson...
8    [{'id_str': '1354061126', 'name': 'Shaun McMah...
9    [{'id_str': '1527485338368610304', 'name': 'りゆ...
Name: mentionedUsers, dtype: str


In [25]:
G = nx.Graph()

# Reply Edges

reply_edges = df[['user', 'in_reply_to_user_id_str']].dropna()

for _, row in reply_edges.iterrows():
    G.add_edge(row['user'], row['in_reply_to_user_id_str'], type='reply')

# Mention Edges

def extract_mentions(x):
    if pd.isna(x):
        return []
    try:
        users = ast.literal_eval(x)
        return [u['screen_name'] for u in users if 'screen_name' in u]
    except:
        return []

df['mentions_list'] = df['mentionedUsers'].apply(extract_mentions)

for _, row in df.iterrows():
    user = row['user']
    for mentioned in row['mentions_list']:
        G.add_edge(user, mentioned, type='mention')


# Hashtag Edges

def extract_hashtags(x):
    if pd.isna(x):
        return []
    try:
        tags = ast.literal_eval(x)
        return [t['text'] for t in tags if 'text' in t]
    except:
        return []

df['hashtags_list'] = df['hashtags'].apply(extract_hashtags)

for _, row in df.iterrows():
    user = row['user']
    for tag in row['hashtags_list']:
        G.add_edge(user, "hashtag_" + tag.lower(), type='hashtag')


# Coversation Edges

for _, row in df.iterrows():
    user = row['user']
    conv = row['conversationId']
    G.add_edge(user, "conv_" + str(conv), type='conversation')


In [26]:
reply_count = 0
mention_count = 0
hashtag_count = 0
conversation_count = 0

for u, v, data in G.edges(data=True):
    if data['type'] == 'reply':
        reply_count += 1
    elif data['type'] == 'mention':
        mention_count += 1
    elif data['type'] == 'hashtag':
        hashtag_count += 1
    elif data['type'] == 'conversation':
        conversation_count += 1

In [27]:
user_nodes = 0
hashtag_nodes = 0
conversation_nodes = 0

for node in G.nodes():
    node = str(node)
    
    if node.startswith("hashtag_"):
        hashtag_nodes += 1
    elif node.startswith("conv_"):
        conversation_nodes += 1
    else:
        user_nodes += 1

In [28]:
import plotly.express as px
import pandas as pd

edge_df = pd.DataFrame({
    "edge_type": ["Reply", "Mention", "Hashtag", "Conversation"],
    "count": [reply_count, mention_count, hashtag_count, conversation_count]
})

fig = px.pie(
    edge_df,
    names='edge_type',
    values='count',
    hole=0.4,
    title='Graph Edge Type Distribution'
)

fig.update_traces(
    textinfo='label+percent+value'
)

fig.show()

In [29]:
print("GRAPH SUMMARY")
print("----------------------")
print("Reply edges:", reply_count)
print("Mention edges:", mention_count)
print("Hashtag edges:", hashtag_count)
print("Conversation edges:", conversation_count)
print()
print("Total edges:", G.number_of_edges())
print("Total nodes:", G.number_of_nodes())
print()
print("User nodes:", user_nodes)
print("Hashtag nodes:", hashtag_nodes)
print("Conversation nodes:", conversation_nodes)
print()
print("Average degree:", sum(dict(G.degree()).values()) / G.number_of_nodes())
print("Largest connected component size:", len(max(nx.connected_components(G), key=len)))

GRAPH SUMMARY
----------------------
Reply edges: 173323
Mention edges: 293840
Hashtag edges: 36475
Conversation edges: 241197

Total edges: 744835
Total nodes: 423345

User nodes: 301104
Hashtag nodes: 10844
Conversation nodes: 111397

Average degree: 3.5188085367726085
Largest connected component size: 279373


In [30]:
import pickle

with open("../data/user_graph.pkl", "wb") as f:
    pickle.dump(G, f)